# FastSpeech2 — Bulgarian single-speaker · **REAL training run**

End-to-end pipeline tuned for a multi-day run: **setup → CONFIG → subset → MFA → preprocess → train (with checkpoints) → resume → synthesize**.

**How to use**
1. Runtime → *Change runtime type* → **GPU** (T4 is fine).
2. Upload your dataset zip to **MyDrive** (Drive root). It can have **any name** — you set that name in the **CONFIG** cell (section 5).
3. Run the cells top to bottom. Section **1 restarts the runtime** once — after the restart, continue from section 2.
4. Everything you tweak (dataset name, batch size, steps, subset size) lives in **one CONFIG cell**.

> Checkpoints are written into Google Drive every `SAVE_STEP` steps, so a disconnect during the 2–3 day run never loses progress — you just re-attach and continue from the last checkpoint (sections 13 / 15).

In [ ]:
!nvidia-smi

## 1. Install MFA via condacolab
⚠️ This **restarts the runtime automatically**. After the restart, *skip this cell* and continue from section 2.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()  # runtime restarts here

## 2. Install MFA + Python dependencies *(run after the restart)*
The `conda install` of MFA pulls Kaldi and can take a few minutes.

In [ ]:
import condacolab
condacolab.check()
# Montreal Forced Aligner (Kaldi-based; conda only)
!conda install -y -c conda-forge montreal-forced-aligner
# FastSpeech2 Python deps (modern versions; the repo code was patched for them)
!pip install -q torch torchaudio numpy scipy librosa pyworld tgt scikit-learn matplotlib tensorboard pyyaml tqdm soundfile

## 3. Mount Google Drive
Holds the dataset zip and all persistent backups + checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone the project

In [ ]:
%cd /content
!rm -rf FastSpeech2
!git clone https://github.com/StankoI/FastSpeech2.git
%cd /content/FastSpeech2

## 5. ⚙️ CONFIG — **edit everything here**
This is the only cell you normally change. It patches the YAML configs in place, so every step below picks up your values. Re-run it any time you want to change e.g. the **batch size**, then relaunch training.

In [ ]:
import re, os

# ============================ EDIT ME ============================
# --- dataset (built by RealData/prepare_realdata.ipynb) ---
DATASET_ZIP   = "bg_realdata.zip"             # exact filename in MyDrive (Drive root)
DATASET_DIR   = "."                           # unzip target; the zip nests everything under bg_realdata/
MANIFEST_PATH = "bg_realdata/manifest.csv"    # cleaned manifest (id|wav_path|text) after unzip

# --- how much data to use ---
TRAIN_HOURS   = 40.0    # hours fed to the WHOLE pipeline (prepare_align + preprocess + training)
MFA_HOURS     = 2.0     # hours used ONLY to train the grapheme aligner; it then aligns all TRAIN_HOURS

# --- training knobs (A100 40 GB) ---
BATCH_SIZE    = 96        # A100-40GB sweet spot (T4 fit 48 in 13 GB). Lower if you ever hit CUDA OOM.
TOTAL_STEP    = 200000    # total training steps
SAVE_STEP     = 2500      # write a checkpoint to Drive every N steps
MAX_SEQ_LEN   = 1400      # must be >= max mel frames; a 15 s clip = 15*22050/256 ~= 1292 frames

# --- where backups + checkpoints live (survives disconnects) ---
DRIVE_DIR     = "/content/drive/MyDrive/fs2_bg_real"
# ================================================================

PRE = "config/Bulgarian/preprocess.yaml"
TRN = "config/Bulgarian/train.yaml"
MOD = "config/Bulgarian/model.yaml"

def _patch(path, pattern, repl):
    """Swap a single value in a YAML file, keeping comments + layout intact."""
    s = open(path, encoding="utf-8").read()
    s2, n = re.subn(pattern, repl, s, flags=re.M)
    assert n == 1, f"expected 1 match for {pattern!r} in {path}, got {n}"
    open(path, "w", encoding="utf-8").write(s2)

_patch(PRE, r'(manifest_path:\s*")[^"]*(")', r'\g<1>' + MANIFEST_PATH + r'\g<2>')
_patch(TRN, r'(batch_size:\s*)\d+', r'\g<1>' + str(BATCH_SIZE))
_patch(TRN, r'(total_step:\s*)\d+', r'\g<1>' + str(TOTAL_STEP))
_patch(TRN, r'(save_step:\s*)\d+',  r'\g<1>' + str(SAVE_STEP))
_patch(MOD, r'(max_seq_len:\s*)\d+', r'\g<1>' + str(MAX_SEQ_LEN))

os.makedirs(DRIVE_DIR, exist_ok=True)
print("preprocess.yaml  manifest_path ->", MANIFEST_PATH)
print("train.yaml       batch_size =", BATCH_SIZE, "| total_step =", TOTAL_STEP, "| save_step =", SAVE_STEP)
print("model.yaml       max_seq_len =", MAX_SEQ_LEN)
print("subset           train on", TRAIN_HOURS, "h | MFA aligner trained on", MFA_HOURS, "h")
print("Drive backups/checkpoints   ->", DRIVE_DIR)


## 6. Unpack the dataset from Drive
Unzips your dataset to `DATASET_DIR` and checks that the manifest and the first wav resolve.

In [ ]:
!cp "/content/drive/MyDrive/{DATASET_ZIP}" .
!unzip -q -o "{DATASET_ZIP}" -d "{DATASET_DIR}"

import os
print("manifest exists:", os.path.exists(MANIFEST_PATH))
first = open(MANIFEST_PATH, encoding="utf-8").readline().split("|")[1]
print("first wav resolves:", first, "->", os.path.exists(first))

## 7. Unzip the HiFi-GAN universal vocoder
The loader expects `hifigan/generator_universal.pth.tar` (the repo ships it zipped).

In [ ]:
!unzip -o hifigan/generator_universal.pth.tar.zip -d hifigan/
!ls -la hifigan/*.pth.tar

## 8. 🎚 Choose how much data to use

Two **independent** knobs (set in the CONFIG cell):

* **`TRAIN_HOURS` (40 h)** — the data that flows through the *whole* pipeline: `prepare_align` → preprocess → FastSpeech2 training.
* **`MFA_HOURS` (2 h)** — a small subset (drawn from the 40 h) used **only to train the grapheme aligner**. That aligner then aligns *all* 40 h in the next section, so every training clip gets durations **without** running the expensive `mfa train` on the full set.

Set `USE_SUBSET = False` to send the full manifest through training (MFA still trains on just `MFA_HOURS`).


In [ ]:
# add the dataset-folder prefix so wav paths resolve from the repo root
lines = [l for l in open(MANIFEST_PATH, encoding="utf-8").read().splitlines() if l.strip()]
out = []
for l in lines:
    a = l.split("|")
    if not a[1].startswith("bg_realdata/"):
        a[1] = "bg_realdata/" + a[1]
    out.append("|".join(a))
open(MANIFEST_PATH, "w", encoding="utf-8").write("\n".join(out) + "\n")

import os
first = out[0].split("|")[1]
print("first wav:", first, "-> exists:", os.path.exists(first))   # expect True

In [ ]:
import os, random, soundfile as sf

# ----------------------- subset settings -----------------------
USE_SUBSET   = True       # False -> whole pipeline uses the FULL manifest (MFA still uses MFA_HOURS)
SUBSET_SEED  = 42         # reproducible random pick
# (TRAIN_HOURS and MFA_HOURS come from the CONFIG cell)
# ---------------------------------------------------------------

TRAIN_MANIFEST = "filelists/train_subset.csv"   # the TRAIN_HOURS subset (drives the whole pipeline)
MFA_LIST       = "filelists/mfa_ids.txt"        # clip ids used ONLY to train the MFA aligner
random.seed(SUBSET_SEED)

rows = [l.rstrip("\n") for l in open(MANIFEST_PATH, encoding="utf-8") if l.strip()]
print(f"full manifest: {len(rows)} clips")

def dur_of(row):
    wav = row.split("|")[1]
    info = sf.info(wav)
    return info.frames / info.samplerate

random.shuffle(rows)

# 1) pick TRAIN_HOURS for the whole pipeline (remember each clip's duration as we go)
train, t_tot, budget = [], 0.0, TRAIN_HOURS * 3600
for r in rows:
    try:
        d = dur_of(r)
    except Exception:
        continue
    train.append((r, d)); t_tot += d
    if USE_SUBSET and t_tot >= budget:
        break
print(f"TRAIN subset       : {len(train)} clips ~ {t_tot/3600:.2f} h")

# 2) from the TRAIN subset, take MFA_HOURS to train the aligner
mfa_ids, m_tot = [], 0.0
for r, d in train:
    mfa_ids.append(r.split("|")[0]); m_tot += d
    if m_tot >= MFA_HOURS * 3600:
        break
print(f"MFA aligner subset : {len(mfa_ids)} clips ~ {m_tot/3600:.2f} h")

with open(TRAIN_MANIFEST, "w", encoding="utf-8") as f:
    f.write("\n".join(r for r, _ in train) + "\n")
with open(MFA_LIST, "w", encoding="utf-8") as f:
    f.write("\n".join(mfa_ids) + "\n")

# the whole pipeline (prepare_align / preprocess / train) uses the TRAIN subset
active = TRAIN_MANIFEST
_patch(PRE, r'(manifest_path:\s*")[^"]*(")', r'\g<1>' + active + r'\g<2>')
print("active manifest ->", active)


## 9. prepare_align — resample to 22.05 kHz and write `.wav`/`.lab` for MFA
Reads the **active** manifest (full or subset) and writes the MFA input tree. The `rm` clears any stale clips from an earlier run so MFA only sees the current selection.

In [ ]:
!rm -rf raw_data/Bulgarian/Bulgarian
!python prepare_align.py config/Bulgarian/preprocess.yaml
!ls raw_data/Bulgarian/Bulgarian | head
!echo "clips written:" && find raw_data/Bulgarian/Bulgarian -name '*.wav' | wc -l

## 10. MFA — train a grapheme aligner on `MFA_HOURS`, then align all `TRAIN_HOURS`

`mfa train` on the small subset produces `bg_acoustic_model.zip`; `mfa align` then uses that model to write a TextGrid for **every** training clip (all `TRAIN_HOURS`). This keeps the slow acoustic-model training tiny while still giving durations for the full training set.

If the alignments look rough later (bad durations → buzzy audio), raise `MFA_HOURS` (e.g. 5–10 h) and re-run this section — more data makes a better aligner.


In [ ]:
# --- Build a small MFA-training corpus (MFA_HOURS) from the ids picked in the subset cell ---
import os, shutil
RAW    = "raw_data/Bulgarian/Bulgarian"           # all TRAIN_HOURS clips (.wav/.lab) from prepare_align
MFADIR = "raw_data/Bulgarian_mfatrain/Bulgarian"  # the small corpus used only to TRAIN the aligner

shutil.rmtree("raw_data/Bulgarian_mfatrain", ignore_errors=True)
os.makedirs(MFADIR, exist_ok=True)
ids = [l.strip() for l in open("filelists/mfa_ids.txt") if l.strip()]
n = 0
for cid in ids:
    w, lab = f"{RAW}/{cid}.wav", f"{RAW}/{cid}.lab"
    if os.path.exists(w) and os.path.exists(lab):
        shutil.copy(w, f"{MFADIR}/{cid}.wav"); shutil.copy(lab, f"{MFADIR}/{cid}.lab"); n += 1
print(f"MFA-train corpus: {n} files (~{MFA_HOURS} h)")

# 1) train the grapheme acoustic model on the small subset -> bg_acoustic_model.zip
!mfa validate raw_data/Bulgarian_mfatrain lexicon/bulgarian-grapheme.txt --clean -j 4 || true
!mfa train raw_data/Bulgarian_mfatrain lexicon/bulgarian-grapheme.txt bg_acoustic_model.zip --clean -j 4

# 2) align the FULL TRAIN_HOURS set with that model -> a TextGrid for every training clip
!mfa align raw_data/Bulgarian lexicon/bulgarian-grapheme.txt bg_acoustic_model.zip preprocessed_data/Bulgarian/TextGrid --clean -j 4
!echo "TextGrids written:" && ls preprocessed_data/Bulgarian/TextGrid/Bulgarian | wc -l


### 💾 Back up MFA outputs to Drive
So a disconnect never forces you to re-run the slow MFA step.

In [ ]:
import os, shutil
os.makedirs(DRIVE_DIR, exist_ok=True)

assert os.path.isdir("preprocessed_data/Bulgarian/TextGrid"), "No TextGrids — did MFA finish?"
shutil.make_archive(f"{DRIVE_DIR}/raw_data_Bulgarian", "zip", "raw_data", "Bulgarian")
if os.path.exists("bg_acoustic_model.zip"):
    shutil.copy("bg_acoustic_model.zip", DRIVE_DIR)

print("backed up MFA outputs ->", DRIVE_DIR)
for f in sorted(os.listdir(DRIVE_DIR)):
    p = os.path.join(DRIVE_DIR, f)
    if os.path.isfile(p):
        print(f"  {f:30s} {os.path.getsize(p)/1e6:8.1f} MB")

## 11. Preprocess — mel / pitch / energy / duration features
Produces `train.txt`, `val.txt`, `stats.json`, `speakers.json` plus the feature `.npy` dirs.

In [ ]:
!python preprocess.py config/Bulgarian/preprocess.yaml
!wc -l preprocessed_data/Bulgarian/train.txt preprocessed_data/Bulgarian/val.txt

### 💾 Back up `preprocessed_data` to Drive *(for fast resume)*
This zip contains everything training needs (`train.txt`, `stats.json`, the feature dirs, and the TextGrids). On resume you just restore it instead of re-running MFA + preprocess.

In [ ]:
import shutil
shutil.make_archive(f"{DRIVE_DIR}/preprocessed_Bulgarian", "zip", "preprocessed_data", "Bulgarian")
print("backed up ->", f"{DRIVE_DIR}/preprocessed_Bulgarian.zip")

## 12. Persist checkpoints to Drive
Symlink `output/` into Drive so checkpoints + logs survive a disconnect. After this, every checkpoint the trainer writes lands directly in Drive.

In [ ]:
import os
os.makedirs(f"{DRIVE_DIR}/output", exist_ok=True)
!rm -rf output
!ln -s "{DRIVE_DIR}/output" output
!ls -la output

## 13. ▶️ Train — **two options**

```
#  OPTION A  ->  START TRAINING FROM ZERO   (fresh model, begins at step 0)
#  OPTION B  ->  CONTINUE FROM A CHECKPOINT  (resume the 2–3 day run)
```

Run the **helper** cell first to see which checkpoints already exist, then pick A or B. Checkpoints are written to Drive every `SAVE_STEP` steps (set in CONFIG).

In [ ]:
# Helper: list checkpoints already saved in Drive and find the latest step.
import os, re
ckpt_dir = "output/ckpt/Bulgarian"
steps = []
if os.path.isdir(ckpt_dir):
    steps = sorted(int(m.group(1)) for f in os.listdir(ckpt_dir)
                   if (m := re.match(r"(\d+)\.pth\.tar$", f)))
LATEST_STEP = steps[-1] if steps else 0
print("checkpoints found:", steps if steps else "none yet")
print("latest step:", LATEST_STEP)

### ▶️ Option A — start training **from zero**
Use this the **first** time only. It ignores any existing checkpoints and begins at step 0.

In [ ]:
# === IF YOU ARE STARTING TRAINING FROM ZERO -> run this cell ===
!MPLBACKEND=Agg python train.py \
    -p config/Bulgarian/preprocess.yaml \
    -m config/Bulgarian/model.yaml \
    -t config/Bulgarian/train.yaml

### ⏯️ Option B — continue **from a checkpoint**
Use this after a disconnect, or whenever you stop and want to keep going. Set the step to resume from (defaults to the latest one found above).

In [ ]:
# === IF YOU ARE CONTINUING TRAINING FROM A CHECKPOINT -> run this cell ===
RESTORE_STEP = LATEST_STEP        # or type an exact step, e.g. 25000

assert RESTORE_STEP > 0, "RESTORE_STEP is 0 -> nothing to resume; use Option A to start from zero."
print("resuming from step", RESTORE_STEP)
!MPLBACKEND=Agg python train.py --restore_step {RESTORE_STEP} \
    -p config/Bulgarian/preprocess.yaml \
    -m config/Bulgarian/model.yaml \
    -t config/Bulgarian/train.yaml

## 14. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir output/log/Bulgarian

## 15. 🔁 Resume after a disconnect

Colab dropped the runtime mid-training? Nothing is lost — checkpoints are in Drive. To continue:

1. Re-run **section 1** (condacolab → restarts), then **section 2** (deps).
2. **Section 3** (mount Drive) · **Section 4** (clone) · **Section 5** (CONFIG — use the *same* values).
3. **Section 7** (vocoder).
4. **This cell** — restore `preprocessed_data` from Drive (skips MFA + preprocess entirely).
5. **Section 12** (re-symlink `output/` to Drive — brings the checkpoints back).
6. **Section 13 → Option B** to continue from the latest step.

In [ ]:
import os, shutil
# Restore the features/alignments training needs (train.txt, stats.json, TextGrids, npy dirs).
shutil.unpack_archive(f"{DRIVE_DIR}/preprocessed_Bulgarian.zip", "preprocessed_data")
print("restored preprocessed_data:",
      "train.txt", os.path.isfile("preprocessed_data/Bulgarian/train.txt"),
      "| stats.json", os.path.isfile("preprocessed_data/Bulgarian/stats.json"))

## 16. 🔊 Synthesize — load a checkpoint from Drive and test text

**To synthesize on a fresh runtime** (e.g. just to hear the latest model), run:
sections **3** (mount) → **4** (clone) → **5** (CONFIG) → **7** (vocoder) → **12** (symlink `output/` so the Drive checkpoints appear) → the **restore** cell just below (needs `stats.json`) → the **synthesize** cell.

If you are already set up from training, skip straight to the synthesize cell.

In [ ]:
# Make sure the stats/speakers files exist (needed for inference). Safe to re-run.
import os, shutil
if not os.path.isfile("preprocessed_data/Bulgarian/stats.json"):
    shutil.unpack_archive(f"{DRIVE_DIR}/preprocessed_Bulgarian.zip", "preprocessed_data")
print("stats.json present:", os.path.isfile("preprocessed_data/Bulgarian/stats.json"))
# show which checkpoint steps are available in Drive
import re
ckpt_dir = "output/ckpt/Bulgarian"
steps = sorted(int(m.group(1)) for f in os.listdir(ckpt_dir)
               if (m := re.match(r"(\d+)\.pth\.tar$", f))) if os.path.isdir(ckpt_dir) else []
print("available checkpoint steps:", steps if steps else "none")

In [ ]:
# ----------------------- edit these two -----------------------
SYNTH_TEXT = "Здравей! Как си днес?"
SYNTH_STEP = 5000          # checkpoint step to load (must be in the list above)
# --------------------------------------------------------------

!MPLBACKEND=Agg python synthesize.py --text "{SYNTH_TEXT}" --restore_step {SYNTH_STEP} \
    --mode single -p config/Bulgarian/preprocess.yaml \
    -m config/Bulgarian/model.yaml -t config/Bulgarian/train.yaml

# play the freshest result inline
import glob, os
from IPython.display import Audio, display
wavs = sorted(glob.glob("output/result/Bulgarian/*.wav"), key=os.path.getmtime)
if wavs:
    print("playing:", wavs[-1]); display(Audio(wavs[-1]))
else:
    print("no wav produced - check the synthesize.py output above")